# dim_club.py

## Purpose
Builds `dim_club` as an SCD2 dimension that tracks club name and league changes.

## Sources
- Bronze: `fifa.bronze.ea_fc_players_raw` (columns `club`, `league`, `gender`)
- Silver: `fifa.silver.dim_gender`, `fifa.silver.dim_league`

## Strategy
- SCD2 with MERGE
- Business key: `club_code` + `gender_id` + `league_id`
  - The `gender_id` distinguishes men's and women's clubs with the same name.
  - The `league_id` distinguishes clubs with the same name in different leagues (e.g. "Nacional" in Uruguay and Portugal).
- Surrogate key: `xxhash64(club_name + gender_id + league_id)`
- Expires the current record when `club_name` changes

## Output
- Table: `fifa.silver.dim_club`

## Columns
- `club_id` (INT)
- `club_code`, `club_name` (STRING)
- `gender_id` (INT): FK to `dim_gender`
- `league_id` (INT): FK to `dim_league`
- `Start_date`, `End_date`, `Is_current` (SCD2 control columns)
- `_ingestion_timestamp` (TIMESTAMP)

## Why the composite business key?

Two clubs can share the same name in different contexts:

- **Same name, different gender**: "FC Barcelona" (men's) and "FC Barcelona" (women's).
- **Same name, different league**: "Nacional" (Uruguay) and "Nacional" (Portugal).

Using only `club_name` would cause collisions. The composite key `club_name + gender_id + league_id` guarantees uniqueness.

## Known Limitation: League Changes

The business key of `dim_club` includes `league_id`:


In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, lit, regexp_extract, when, concat,trim, explode, split, pmod, xxhash64
from delta.tables import DeltaTable

# ===== Read Bronze =====
bronze = spark.read.table("fifa.bronze.ea_fc_players_raw")


clubs = (
    bronze
    .select(
        col("club").alias("club_name"),
        col("league").alias("league_name"),
        col("gender").alias("gender")
    )
    .filter(col("club_name").isNotNull() & (col("club_name") != ""))
    .filter(col("league_name").isNotNull() & (col("league_name") != ""))
    .select(
        trim(col("club_name")).alias("club_name"),
        trim(col("league_name")).alias("league_name"),
        trim(col("gender")).alias("gender")
    )
    .distinct()
)


# Special case: disambiguate "Nacional" (exists in Portugal and Uruguay)
CLUB_NAME_FIXES = {
    ("Nacional", "Primeira Liga"): "Nacional (Portugal)",
    ("Nacional", "Primera División"): "Nacional (Uruguay)",
}

for (club, league), new_name in CLUB_NAME_FIXES.items():
    clubs = clubs.withColumn(
        "club_name",
        when(
            (col("club_name") == club) & (col("league_name") == league),
            lit(new_name)
        ).otherwise(col("club_name"))
    )

# ===== Enrich with dimensions =====
dim_gender = spark.read.table("fifa.silver.dim_gender")
dim_league = spark.read.table("fifa.silver.dim_league").filter(col("Is_current") == True)

clubs_enriched = (
    clubs
    .join(dim_gender.select("gender_id", "gender_name"), 
          clubs.gender == dim_gender.gender_name, how="left")
    .join(dim_league.select("league_id", "league_name"), 
          on="league_name", how="left")
)

# ===== Build SCD2-ready DataFrame =====
new_clubs = (
    clubs_enriched
    .withColumn("club_key", concat(col("club_name"), lit("|"), col("gender_id"), lit("|"), col("league_id")))
    .withColumn("club_id", pmod(xxhash64(col("club_key")), 1000000).cast("int"))
    .withColumn("club_code", col("club_name"))
    .withColumn("Start_date", current_timestamp())
    .withColumn("End_date", lit(None).cast("timestamp"))
    .withColumn("Is_current", lit(True))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .select(
        "club_id", "club_code", "club_name", "gender_id", "league_id",
        "Start_date", "End_date", "Is_current", "_ingestion_timestamp"
    )
)
# ===== Create table Silver =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.dim_club (
    club_id INT,
    club_code STRING,
    club_name STRING,
    gender_id INT,
    league_id INT,
    Start_date TIMESTAMP,
    End_date TIMESTAMP,
    Is_current BOOLEAN,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA;
""")

# MERGE SCD2

target = DeltaTable.forName(spark, "fifa.silver.dim_club")

(
    target.alias("t")
    .merge(
        new_clubs.alias("s"),
        "t.club_code = s.club_code AND t.gender_id = s.gender_id  AND t.Is_current = true"
    )
    .whenMatchedUpdate(
        condition="t.club_name <> s.club_name OR t.league_id <> s.league_id", 
        set={
            "End_date": "s.Start_date",
            "Is_current": "false"
        }
    )
    .whenNotMatchedInsert(
        values={
            "club_id": "s.club_id",
            "club_code": "s.club_code",
            "club_name": "s.club_name",
            "gender_id": "s.gender_id",
            "league_id": "s.league_id",
            "Start_date": "s.Start_date",
            "End_date": "s.End_date",
            "Is_current": "s.Is_current",
            "_ingestion_timestamp": "s._ingestion_timestamp"
        }
    )
    .execute()
)

print(f"dim_club actualizada con {spark.read.table('fifa.silver.dim_club').count()} filas")

LIMIT 10 of the dim_club table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
SELECT * FROM fifa.silver.dim_club limit 10;